<h1>An elevator built with object-oriented programming.</h1>
<h4>Blake Rayvid - <a href=https://github.com/brayvid>https://github.com/brayvid</a></h4>

In [16]:
import time # For simulating door open/close time

class Building:
  def __init__(self, num_floors, num_elevators):
    if num_floors < 2:
      raise Exception("Building must have at least 2 floors.")
    if num_elevators < 1:
      raise Exception("Building must have at least 1 elevator.")

    self.num_floors = num_floors
    self.num_elevators = num_elevators
    self.occupants = [] # All riders currently in the building system

    self.floors = [Lobby(floor_number = 0, building = self)]
    for i in range(1, num_floors-1):
      self.floors.append(MiddleFloor(floor_number = i, building = self))
    self.floors.append(Roof(floor_number = num_floors-1, building = self))

    self.elevators = []
    for i in range(num_elevators):
      self.elevators.append(Elevator(building=self, elevator_id=i)) # Pass ID

    # Centralized request queues: set of (floor_number, "UP"/"DOWN") tuples
    self.floor_requests = set() # e.g., {(2, "UP"), (5, "DOWN")}

  def add_occupant(self, rider):
    if rider not in self.occupants:
      self.occupants.append(rider)

  def remove_occupant(self, rider):
    if rider in self.occupants:
      self.occupants.remove(rider)

  def add_floor_request(self, floor_number, direction): # direction is "UP" or "DOWN"
    request = (floor_number, direction)
    if request not in self.floor_requests:
        self.floor_requests.add(request)
        print(f"Building: Received floor request at F{floor_number} for {direction}.")
        # Attempt to dispatch immediately (or could be done in tick)
        self._dispatch_request(request)
    else:
        # print(f"Building: Floor request at F{floor_number} for {direction} already exists.")
        pass


  def clear_floor_request(self, floor_number, direction_str):
      request = (floor_number, direction_str)
      if request in self.floor_requests:
          self.floor_requests.remove(request)
          print(f"Building: Cleared floor request {request}.")
          # Reset the actual button on the floor object
          floor_obj = self.floors[floor_number]
          if direction_str == "UP" and hasattr(floor_obj, 'reset_up_button'):
              floor_obj.reset_up_button()
          elif direction_str == "DOWN" and hasattr(floor_obj, 'reset_down_button'):
              floor_obj.reset_down_button()

  def _dispatch_request(self, request_tuple):
    """Assigns an unassigned floor request to the best available elevator."""
    # Simple dispatch: find the first IDLE elevator.
    # If none idle, find one moving towards the request and can stop.
    # This is a basic example. More sophisticated dispatching is possible.

    req_floor, req_dir_str = request_tuple
    req_dir_val = Elevator.UP if req_dir_str == "UP" else Elevator.DOWN

    best_elevator = None
    min_score = float('inf') # Lower score is better

    # Try to find an elevator to serve this request
    for elev in self.elevators:
        # Score 1: Idle elevator at the floor (perfect match)
        if elev.direction == Elevator.IDLE and elev.current_floor == req_floor:
            best_elevator = elev
            break # Perfect match

        # Score 2: Idle elevator, not at floor (distance based)
        if elev.direction == Elevator.IDLE:
            distance = abs(elev.current_floor - req_floor)
            if distance < min_score: # Prioritize closer idle elevators
                min_score = distance
                best_elevator = elev
            continue # Check other elevators for potentially better idle match or active match

        # Score 3: Elevator moving towards the request floor and in the same direction
        is_moving_towards = (elev.direction == Elevator.UP and elev.current_floor <= req_floor) or \
                            (elev.direction == Elevator.DOWN and elev.current_floor >= req_floor)

        if elev.direction == req_dir_val and is_moving_towards:
            distance = abs(elev.current_floor - req_floor)
            # Add a penalty if it's already serving many requests or is far
            # For simplicity, just distance for now
            score = distance + 5 # Add penalty for active elevator vs idle
            if score < min_score:
                min_score = score
                best_elevator = elev

    if best_elevator:
      print(f"Building: Dispatching request {request_tuple} to Elevator {best_elevator.id}")
      best_elevator.assign_pickup_request(req_floor, req_dir_str)
      # The request is "claimed" by an elevator, but don't remove from building.floor_requests
      # until the elevator *actually* services it and tells the building.
      # For now, let's assume assign_pickup_request means it's "taken"
      # and will be cleared from building.floor_requests when elevator services it.
      # This means `_dispatch_request` might be called for requests already assigned.
      # To prevent re-dispatching, we can check if any elevator already has this pickup.
      # A better way: elevators pick from building.floor_requests in their update.
      # For now, this simple dispatch is OK. We'll let the elevator clear it from building.
    else:
      print(f"Building: No suitable elevator found immediately for {request_tuple}. Will retry or wait.")


  def tick(self):
    """Advances the simulation by one time step."""
    print(f"\n--- Tick ---")

    # 1. Update all elevators
    for elevator in self.elevators:
      elevator.update()

    # 2. Re-evaluate unassigned floor requests (if any weren't assigned immediately)
    #    Or, if an elevator became idle, it could try to pick one up.
    #    For this version, `add_floor_request` tries to dispatch immediately.
    #    A more robust system might have elevators poll for tasks or a central dispatcher
    #    running in the tick.
    # Let's try dispatching any remaining requests:
    # Make a copy because `_dispatch_request` might indirectly modify it (if it removed)
    # but `assign_pickup_request` only adds to elevator's internal list.
    # The elevator itself will call `building.clear_floor_request` when done.
    for req in list(self.floor_requests):
        # Check if this request is already being handled by an elevator
        is_handled = False
        for elev in self.elevators:
            if req in elev.pickup_requests:
                is_handled = True
                break
        if not is_handled:
            # print(f"Building: Attempting to dispatch unhandled request {req} during tick.")
            self._dispatch_request(req)


    # 3. Optional: Rider behavior (e.g., new riders arriving, riders deciding to leave)
    #    For now, this is handled by explicit calls like Rider.request_elevator()

    self.status_report()


  def status_report(self):
    print("Building Status:")
    print(f"  Floor Requests: {self.floor_requests}")
    for floor in self.floors:
      waiting_riders_ids = [r.id for r in floor.waiting]
      up_pressed = " (UP)" if hasattr(floor, 'up_button') and floor.up_button.pressed else ""
      down_pressed = " (DOWN)" if hasattr(floor, 'down_button') and floor.down_button.pressed else ""
      print(f"  Floor {floor.floor_number}: Waiting: {waiting_riders_ids}{up_pressed}{down_pressed}")
    for elevator in self.elevators:
      print(f"  {elevator}")
    print(f"  Occupants in building: {[r.id for r in self.occupants]}")
    print("--- End Tick ---")

class Floor:
  def __init__(self, floor_number, building):
    if floor_number < 0:
      raise Exception("'floor_number' must be a non-negative integer.")
    self.floor_number = floor_number
    self.building = building
    self.waiting = [] # Riders waiting on this floor
    # Buttons will be added by subclasses

  def add_rider_waiting(self, rider):
    if rider not in self.waiting:
      self.waiting.append(rider)

  def remove_rider_waiting(self, rider):
    if rider in self.waiting:
      self.waiting.remove(rider)

  # To be implemented by subclasses
  def press_up_button(self):
    pass

  def press_down_button(self):
    pass

  def is_up_button_pressed(self):
    return False

  def is_down_button_pressed(self):
    return False

  def reset_up_button(self):
    pass

  def reset_down_button(self):
    pass

class Lobby(Floor):
  def __init__(self, floor_number, building):
    super().__init__(floor_number = floor_number, building = building)
    self.up_button = UpDownButton(up=True, floor=self)

  def press_up_button(self):
    self.up_button.push()
    self.building.add_floor_request(self.floor_number, direction="UP")

  def is_up_button_pressed(self):
    return self.up_button.pressed

  def reset_up_button(self):
    self.up_button.reset()

class Roof(Floor):
  def __init__(self, floor_number, building):
    super().__init__(floor_number = floor_number, building = building)
    self.down_button = UpDownButton(up=False, floor=self)

  def press_down_button(self):
    self.down_button.push()
    self.building.add_floor_request(self.floor_number, direction="DOWN")

  def is_down_button_pressed(self):
    return self.down_button.pressed

  def reset_down_button(self):
    self.down_button.reset()


class MiddleFloor(Floor):
  def __init__(self, floor_number, building):
    super().__init__(floor_number = floor_number, building = building)
    self.up_button = UpDownButton(up=True, floor=self)
    self.down_button = UpDownButton(up=False, floor=self) # Corrected typo: down_buton -> down_button

  def press_up_button(self):
    self.up_button.push()
    self.building.add_floor_request(self.floor_number, direction="UP")

  def press_down_button(self):
    self.down_button.push() # was self.down_buton
    self.building.add_floor_request(self.floor_number, direction="DOWN")

  def is_up_button_pressed(self):
    return self.up_button.pressed

  def is_down_button_pressed(self):
    return self.down_button.pressed # was self.down_buton

  def reset_up_button(self):
    self.up_button.reset()

  def reset_down_button(self):
    self.down_button.reset() # was self.down_buton

class Elevator:
  IDLE = 0
  UP = 1
  DOWN = -1
  DOOR_OPEN_DURATION = 2 # seconds or ticks for doors to be open

  def __init__(self, building, elevator_id, weight_limit = 2000): # Add elevator_id
    self.id = elevator_id # Unique identifier
    self.weight_limit = weight_limit
    self.building = building
    self.current_floor = 0
    self.current_weight = 0
    self.occupants = []
    self.direction = Elevator.IDLE # UP, DOWN, IDLE

    # Internal requests from riders inside this elevator
    self.destination_requests = set() # Set of floor numbers

    # External calls assigned to this elevator by the building
    self.pickup_requests = set() # Set of (floor_number, direction_string) tuples e.g. (3, "UP")

    self.buttons = [] # Destination buttons inside the elevator
    for i in range(self.building.num_floors):
      self.buttons.append(DestButton(floor_number=i, elevator=self))

    self.doors_open = False
    self.door_timer = 0 # How long doors have been open

  def __repr__(self):
    return (f"Elevator {self.id} (F{self.current_floor}, Dir:{self.direction}, "
            f"Weight:{self.current_weight}/{self.weight_limit}, "
            f"Occ:{len(self.occupants)}, Doors:{'Open' if self.doors_open else 'Closed'}, "
            f"Dests:{sorted(list(self.destination_requests))}, "
            f"Pickups:{sorted(list(self.pickup_requests))})")

  def press_destination_button(self, floor_number):
    if 0 <= floor_number < self.building.num_floors:
      self.buttons[floor_number].push()
      if floor_number != self.current_floor: # No need to request current floor
          self.destination_requests.add(floor_number)
          print(f"Elevator {self.id}: Rider pressed button for floor {floor_number}")

  def assign_pickup_request(self, floor_number, direction_str):
    """Called by Building to assign a floor call to this elevator."""
    self.pickup_requests.add((floor_number, direction_str))
    print(f"Elevator {self.id}: Assigned pickup at F{floor_number} going {direction_str}")
    # If idle, set direction towards this new pickup
    if self.direction == Elevator.IDLE and self.current_floor != floor_number:
        self.direction = Elevator.UP if floor_number > self.current_floor else Elevator.DOWN


  def _should_stop_at_current_floor(self):
    # 1. Is current floor a destination for someone inside?
    if self.current_floor in self.destination_requests:
      return True

    # 2. Is current floor a pickup request matching current direction or if idle?
    for floor, req_direction_str in self.pickup_requests:
        if floor == self.current_floor:
            req_direction_val = Elevator.UP if req_direction_str == "UP" else Elevator.DOWN
            # Stop if idle, or if moving in the direction of the request,
            # or if this is the turnaround point and the request is for the new direction.
            if self.direction == Elevator.IDLE or self.direction == req_direction_val:
                return True
            # Special case: if elevator will change direction here.
            # E.g., moving UP, highest request is current_floor, but there's a DOWN pickup here.
            # This needs more sophisticated logic if combining SCAN with pickup logic.
            # For now, simpler: stop if direction matches or idle.

    return False

  def _handle_boarding_alighting(self):
    # Alighting
    alighted_riders = []
    for rider in list(self.occupants): # Iterate over a copy
      if rider.dest_floor_num == self.current_floor: # CHANGED: dest_floor -> dest_floor_num
        rider.exit_elev(self) # Rider exits, updates floor waiting list
        self.buttons[self.current_floor].reset() # Reset internal button
        alighted_riders.append(rider)
        print(f"Elevator {self.id}: Rider {rider.id if hasattr(rider, 'id') else 'N/A'} exited at floor {self.current_floor}")
    if self.current_floor in self.destination_requests:
        self.destination_requests.remove(self.current_floor)

    # Boarding
    boarded_riders = []
    current_floor_obj = self.building.floors[self.current_floor]
    for rider in list(current_floor_obj.waiting): # Iterate over a copy
      if self.current_weight + rider.weight <= self.weight_limit:
        # Rider wants to go in the elevator's current/intended direction, or elevator is idle
        rider_wants_up = rider.dest_floor_num > self.current_floor    # CHANGED: dest_floor -> dest_floor_num
        rider_wants_down = rider.dest_floor_num < self.current_floor  # CHANGED: dest_floor -> dest_floor_num

        can_board = False
        if self.direction == Elevator.IDLE: # If idle, take anyone
            can_board = True
        elif self.direction == Elevator.UP and rider_wants_up:
            can_board = True
        elif self.direction == Elevator.DOWN and rider_wants_down:
            can_board = True

        pickup_to_clear = None
        if can_board:
            rider_req_direction_str = "UP" if rider_wants_up else "DOWN"
            for pr_floor, pr_dir_str in self.pickup_requests:
                if pr_floor == self.current_floor and pr_dir_str == rider_req_direction_str:
                    pickup_to_clear = (pr_floor, pr_dir_str)
                    break

        if can_board:
          rider.enter(self) # Rider enters, updates elevator occupants
          boarded_riders.append(rider)
          print(f"Elevator {self.id}: Rider {rider.id if hasattr(rider, 'id') else 'N/A'} entered at floor {self.current_floor} for F{rider.dest_floor_num}") # CHANGED: dest_floor -> dest_floor_num
          # Rider presses their destination button inside
          self.press_destination_button(rider.dest_floor_num) # CHANGED: dest_floor -> dest_floor_num

          if pickup_to_clear and pickup_to_clear in self.pickup_requests:
              self.pickup_requests.remove(pickup_to_clear)
              self.building.clear_floor_request(pickup_to_clear[0], pickup_to_clear[1])

    pickups_at_this_floor_in_current_dir = [
        (f, d_str) for f, d_str in self.pickup_requests
        if f == self.current_floor and ((self.direction == Elevator.UP and d_str == "UP") or \
                                       (self.direction == Elevator.DOWN and d_str == "DOWN") or \
                                        self.direction == Elevator.IDLE)
    ]
    for pr in pickups_at_this_floor_in_current_dir:
        if pr in self.pickup_requests:
            # Check if there are still riders waiting for this specific call
            # This is a bit more complex to do perfectly without iterating all riders again
            # For now, if we opened doors and it was a pickup, assume we tried to service it.
            # The removal of pickup_requests upon successful boarding is more precise.
            # This part ensures if an elevator stops for a pickup and no one boards for *that specific direction*
            # (e.g. elevator is full, or rider changed mind), the request might still be cleared.
            # This might need refinement based on desired behavior.
            # For now, the logic above (removing when rider boards) is primary.
            # This below loop will clear general pickups if the elevator stops at the floor
            # in the correct direction.
            pass # The more specific clearing is done when a rider boards.
            # Let's refine: only clear it if no one matching this request is left waiting.

            riders_still_waiting_for_this_pickup = False
            floor_obj = self.building.floors[self.current_floor]
            for waiting_rider in floor_obj.waiting:
                waiting_rider_wants_up = waiting_rider.dest_floor_num > self.current_floor
                waiting_rider_wants_down = waiting_rider.dest_floor_num < self.current_floor
                if pr[1] == "UP" and waiting_rider_wants_up:
                    riders_still_waiting_for_this_pickup = True
                    break
                if pr[1] == "DOWN" and waiting_rider_wants_down:
                    riders_still_waiting_for_this_pickup = True
                    break

            if not riders_still_waiting_for_this_pickup: # Or if the elevator is now empty of matching pickups
                # (This check is actually done implicitly by the building clearing the request
                # when the button is reset. The elevator just removes it from its own list).
                if pr in self.pickup_requests: # Double check it's still there
                    self.pickup_requests.remove(pr)
                    self.building.clear_floor_request(pr[0], pr[1])
                    print(f"Elevator {self.id}: Cleared pickup request {pr} as no more relevant riders are waiting or it was serviced.")

  def _get_next_target_floor(self):
    """Determine the next floor to move towards based on current direction and requests."""
    if not self.destination_requests and not self.pickup_requests:
      return None # No requests at all

    potential_stops = set()
    # Add all internal destinations
    potential_stops.update(self.destination_requests)
    # Add all pickup floors
    for floor, _ in self.pickup_requests:
        potential_stops.add(floor)

    if not potential_stops:
        return None

    if self.direction == Elevator.UP:
      # Stops above current floor
      stops_above = sorted([f for f in potential_stops if f > self.current_floor])
      if stops_above:
        return stops_above[0] # Closest stop above
      # If no stops above, but requests exist below, prepare to turn around
      stops_below = sorted([f for f in potential_stops if f < self.current_floor], reverse=True)
      if stops_below: # Will turn around
          return stops_below[0]
      return None # Only requests at current floor (should have been handled by door opening)

    elif self.direction == Elevator.DOWN:
      # Stops below current floor
      stops_below = sorted([f for f in potential_stops if f < self.current_floor], reverse=True)
      if stops_below:
        return stops_below[0] # Closest stop below
      # If no stops below, but requests exist above, prepare to turn around
      stops_above = sorted([f for f in potential_stops if f > self.current_floor])
      if stops_above: # Will turn around
          return stops_above[0]
      return None

    elif self.direction == Elevator.IDLE:
      # Find the closest request (either pickup or destination)
      if not potential_stops: return None
      closest_floor = -1
      min_dist = self.building.num_floors + 1
      for floor in potential_stops:
          dist = abs(floor - self.current_floor)
          if dist < min_dist:
              min_dist = dist
              closest_floor = floor
          elif dist == min_dist: # Prioritize current floor if tied (e.g. pickup at current floor)
              if floor == self.current_floor:
                  closest_floor = floor
      return closest_floor if closest_floor != -1 else None

    return None # Should not happen

  def update(self):
    #print(f"Updating {self}")
    if self.doors_open:
      self.door_timer += 1
      if self.door_timer >= Elevator.DOOR_OPEN_DURATION:
        self.doors_open = False
        self.door_timer = 0
        print(f"Elevator {self.id}: Doors closing at floor {self.current_floor}")
      else:
        # Doors still open, do nothing else this tick (allow time for boarding/alighting)
        return

    # Check if we should stop at the current floor
    if self._should_stop_at_current_floor():
      if not self.doors_open: # Only open if not already open
        self.doors_open = True
        self.door_timer = 0 # Reset timer
        print(f"Elevator {self.id}: Doors opening at floor {self.current_floor}")
        self._handle_boarding_alighting()
      return # Done for this tick, will handle closing next tick

    # Determine next movement based on requests (SCAN-like)
    next_target = self._get_next_target_floor()

    if next_target is not None:
      if next_target > self.current_floor:
        self.direction = Elevator.UP
        self.current_floor += 1
        print(f"Elevator {self.id}: Moving UP to {self.current_floor}")
      elif next_target < self.current_floor:
        self.direction = Elevator.DOWN
        self.current_floor -= 1
        print(f"Elevator {self.id}: Moving DOWN to {self.current_floor}")
      else: # Target is current floor, means we should stop (handled above)
        self.direction = Elevator.IDLE # Or prepare to change if no more in this dir
        # This case should ideally be fully handled by _should_stop_at_current_floor
        # and subsequent door opening. If we reach here, it might indicate a logic gap.
        # For now, if target is current and doors aren't open, try to open them.
        if not self.doors_open and self._should_stop_at_current_floor():
            self.doors_open = True
            self.door_timer = 0
            print(f"Elevator {self.id}: Doors opening at floor {self.current_floor} (target was current)")
            self._handle_boarding_alighting()

    else: # No target
      if self.direction != Elevator.IDLE:
          print(f"Elevator {self.id}: No more requests in current direction. Becoming IDLE at F{self.current_floor}.")
      self.direction = Elevator.IDLE
      # Optionally, return to lobby if idle for too long (more advanced)

  # ascend/descend are now handled by update() based on logic
  # def ascend(self): ...
  # def descend(self): ...
class Rider:
  _next_id = 0 # Class variable for unique rider IDs

  def __init__(self, building, origin_floor_num=0, dest_floor_num=0, weight=150):
    self.id = Rider._next_id
    Rider._next_id += 1

    self.weight = weight
    self.current_floor_num = origin_floor_num # Where the rider currently IS
    self.dest_floor_num = dest_floor_num     # Where the rider WANTS to go
    self.building = building
    self.elevator = None # Elevator object, not number

    if not (0 <= self.current_floor_num < self.building.num_floors and \
            0 <= self.dest_floor_num < self.building.num_floors):
        raise ValueError("Invalid origin or destination floor for Rider.")

    self.building.floors[self.current_floor_num].add_rider_waiting(self)
    self.building.add_occupant(self) # New method in Building
    print(f"Rider {self.id} created at F{self.current_floor_num}, wants F{self.dest_floor_num}")


  def __repr__(self):
      return f"Rider {self.id} (W:{self.weight}, At:F{self.current_floor_num}, Dest:F{self.dest_floor_num}, InElev:{self.elevator.id if self.elevator else 'None'})"

  def request_elevator(self):
    if self.elevator:
      print(f"Rider {self.id}: Already in elevator {self.elevator.id}, cannot request another.")
      return
    if self.current_floor_num == self.dest_floor_num:
        print(f"Rider {self.id}: Already at destination F{self.current_floor_num}, no request needed.")
        return

    floor_obj = self.building.floors[self.current_floor_num]
    if self.dest_floor_num > self.current_floor_num:
      print(f"Rider {self.id} at F{self.current_floor_num} requests UP call for F{self.dest_floor_num}")
      floor_obj.press_up_button()
    elif self.dest_floor_num < self.current_floor_num:
      print(f"Rider {self.id} at F{self.current_floor_num} requests DOWN call for F{self.dest_floor_num}")
      floor_obj.press_down_button()

  def enter(self, elevator_obj): # Takes elevator object
    if self.current_floor_num != elevator_obj.current_floor:
      raise Exception("Elevator is not at the rider's current floor.")
    if self.weight + elevator_obj.current_weight > elevator_obj.weight_limit:
      raise Exception("Not allowed - Elevator would be over weight limit.")

    self.building.floors[self.current_floor_num].remove_rider_waiting(self)
    elevator_obj.occupants.append(self)
    self.elevator = elevator_obj
    elevator_obj.current_weight += self.weight
    # Rider automatically presses their destination button upon entering
    # self.elevator.press_destination_button(self.dest_floor_num) # This is now done in elevator's board_passengers

  def exit_elev(self, elevator_obj=None): # elevator_obj is optional but good for consistency
    # If elevator_obj is not passed, use self.elevator
    target_elevator = elevator_obj if elevator_obj else self.elevator
    if not target_elevator:
      raise Exception("Rider is not in an elevator.")

    # Ensure rider is actually in this elevator's occupant list
    if self not in target_elevator.occupants:
        # This might happen if exit_elev is called manually without proper context
        print(f"Warning: Rider {self.id} was asked to exit elevator {target_elevator.id} but was not listed as an occupant.")
    else:
        target_elevator.occupants.remove(self)

    self.current_floor_num = target_elevator.current_floor
    self.building.floors[self.current_floor_num].add_rider_waiting(self)
    target_elevator.current_weight -= self.weight
    self.elevator = None
    # If rider reached destination, they might leave the building or make a new request later
    if self.current_floor_num == self.dest_floor_num:
        print(f"Rider {self.id} arrived at destination F{self.current_floor_num}.")
        # Potentially rider could decide to leave building if at lobby, etc.

  def leave_building(self): # Renamed from leave for clarity
    if self.current_floor_num == 0 and not self.elevator: # Lobby and not in an elevator
      self.building.floors[0].remove_rider_waiting(self)
      self.building.remove_occupant(self) # New method in Building
      print(f"Rider {self.id} has left the building from the Lobby.")
    else:
      reason = "not in lobby" if self.current_floor_num != 0 else "in an elevator"
      print(f"Rider {self.id} cannot leave building: {reason}.")

  # 'choose' is effectively setting dest_floor_num, can be part of init or a new method
  def set_new_destination(self, new_dest_floor_num):
      if self.elevator:
          print(f"Rider {self.id}: Cannot change destination while in elevator. (Or implement re-pressing button)")
          return
      if not (0 <= new_dest_floor_num < self.building.num_floors):
          raise ValueError("Invalid new destination floor.")
      print(f"Rider {self.id} at F{self.current_floor_num} now wants to go to F{new_dest_floor_num}")
      self.dest_floor_num = new_dest_floor_num
      self.request_elevator() # Make a new request

class Button:
  def __init__(self):
    self.pressed = False

  def push(self):
    self.pressed = True
    # print(f"Button {self} pushed") # For debugging

  def reset(self):
    self.pressed = False
    # print(f"Button {self} reset") # For debugging

class UpDownButton(Button):
  def __init__(self, up, floor): # Add floor reference
    super().__init__()
    self.up = up # True for up, False for down
    self.floor = floor # Reference to the Floor object

  def __repr__(self):
    return f"UpDownButton(Floor: {self.floor.floor_number}, Dir: {'UP' if self.up else 'DOWN'}, Pressed: {self.pressed})"

class DestButton(Button):
  def __init__(self, floor_number, elevator): # Add elevator reference
    super().__init__()
    self.floor_number = floor_number
    self.elevator = elevator # Reference to the Elevator object

  def __repr__(self):
    return f"DestButton(Floor: {self.floor_number}, Elevator: {self.elevator.id if hasattr(self.elevator, 'id') else 'N/A'}, Pressed: {self.pressed})"

In [17]:
if __name__ == '__main__':
  # Setup
  building = Building(num_floors=10, num_elevators=2)

  # Create some riders
  rider1 = Rider(building, origin_floor_num=0, dest_floor_num=5, weight=100) # Lobby to 5
  rider2 = Rider(building, origin_floor_num=3, dest_floor_num=8, weight=150) # 3 to 8
  rider3 = Rider(building, origin_floor_num=9, dest_floor_num=1, weight=120) # 9 to 1

  # Riders make their initial requests
  rider1.request_elevator()
  rider2.request_elevator()
  rider3.request_elevator()

  # Simulate for a number of ticks
  for i in range(30): # Run for 30 steps
    building.tick()
    # time.sleep(0.5) # Optional: to slow down for readability

    # Example of a new rider appearing later
    if i == 5:
        print("\n!!! New rider appearing !!!")
        rider4 = Rider(building, origin_floor_num=1, dest_floor_num=0, weight=160)
        rider4.request_elevator()

    # Check if all riders have reached destinations and left (if applicable)
    # This part is for a more complete simulation end condition
    all_done = True
    for r in building.occupants:
        if r.elevator is not None or r.current_floor_num != r.dest_floor_num:
            # If rider is in elevator OR not at their destination floor yet
            all_done = False
            break
        # If at destination and it's lobby, they might leave
        if r.current_floor_num == r.dest_floor_num and r.current_floor_num == 0:
            # Simple: if at lobby and destination, they leave
            # A more complex rider AI would make this decision
            if r in building.floors[0].waiting: # ensure they are on the floor, not just logically at dest
                r.leave_building()
                # Note: leave_building removes from building.occupants, so iterating over it
                # while modifying can be tricky. This example is simplified.

    if not building.occupants: # If all riders have left the building
        print("\nAll riders have reached their destinations and left the building (if applicable).")
        break
    if not building.floor_requests and all(not elev.destination_requests and not elev.pickup_requests for elev in building.elevators):
        # And all riders are at their destinations
        all_at_dest = True
        for r in building.occupants:
             if r.current_floor_num != r.dest_floor_num:
                all_at_dest = False
                break
        if all_at_dest:
            print("\nAll requests satisfied and elevators are idle. Riders at destinations.")
            # riders may choose to make new requests or leave
            # for now, end simulation
            break

  print("\nSimulation finished.")
  building.status_report()

Rider 0 created at F0, wants F5
Rider 1 created at F3, wants F8
Rider 2 created at F9, wants F1
Rider 0 at F0 requests UP call for F5
Building: Received floor request at F0 for UP.
Building: Dispatching request (0, 'UP') to Elevator 0
Elevator 0: Assigned pickup at F0 going UP
Rider 1 at F3 requests UP call for F8
Building: Received floor request at F3 for UP.
Building: Dispatching request (3, 'UP') to Elevator 0
Elevator 0: Assigned pickup at F3 going UP
Rider 2 at F9 requests DOWN call for F1
Building: Received floor request at F9 for DOWN.
Building: Dispatching request (9, 'DOWN') to Elevator 1
Elevator 1: Assigned pickup at F9 going DOWN

--- Tick ---
Elevator 0: Doors opening at floor 0
Elevator 0: Rider 0 entered at floor 0 for F5
Elevator 0: Rider pressed button for floor 5
Building: Cleared floor request (0, 'UP').
Elevator 1: Moving UP to 1
Building Status:
  Floor Requests: {(3, 'UP'), (9, 'DOWN')}
  Floor 0: Waiting: []
  Floor 1: Waiting: []
  Floor 2: Waiting: []
  Floor 3